# Envelope GPI on the real, certified DST basis

Everything through "Part 2" below is copied verbatim from `03_adaptive_basis.ipynb`
(DST env, exact tabular dynamics, `sfols_certified`'s Certify-or-Grow basis) so the
envelope-GPI construction below runs on the *actual* certified basis this repo
produces, not a synthetic random MDP. `SFOLS/11_envelope_gpi.ipynb` already validated
Theorem `thm:envelope` (and the Lemma/Remark on the Jensen mechanism behind it) on
random MDPs with zero violations; this notebook checks the same claims on the real
domain, and -- new here -- measures actual wall-clock cost and honest realized-value
comparisons against standard GPI, not just the estimate the theorem is about.

## Part 0/1/2 (verbatim from 03_adaptive_basis.ipynb): DST setup, SF tables, Certify-or-Grow

In [16]:
import heapq
import itertools
import random
import time

import numpy as np
from scipy.optimize import linprog
import mo_gymnasium as mo_gym

np.set_printoptions(suppress=True, precision=3)

env = mo_gym.make("deep-sea-treasure-v0")
core = env.unwrapped
GAMMA = 0.99
N_ACTIONS = core.action_space.n
FEAT_DIM = 2
START_STATE = (0, 0)

ground_truth_ccs = np.array(core.pareto_front(gamma=GAMMA))
print(f"{N_ACTIONS} actions, {FEAT_DIM} objectives, ground-truth CCS has {len(ground_truth_ccs)} points.")

4 actions, 2 objectives, ground-truth CCS has 10 points.


In [17]:
class SFTable:
    def __init__(self, num_actions, feat_dim, default_vec):
        self.default_vec = default_vec 
        self.table = {}

    def __getitem__(self, state):
        if state not in self.table:
            self.table[state] = self.default_vec.copy()
        return self.table[state]

def expected_sf(psi_table, w, start_state=START_STATE):
    values = psi_table[start_state] @ w
    return psi_table[start_state][np.argmax(values)].copy()

In [18]:
def v_smp(psi_vecs, w):
    if not psi_vecs:
        return -np.inf
    return max(v @ w for v in psi_vecs)


def corner_weights_2d(psi_vecs, n_grid=2000):
    w0_grid = np.linspace(0.0, 1.0, n_grid)
    weights = np.stack([w0_grid, 1 - w0_grid], axis=1)
    values_per_policy = weights @ np.array(psi_vecs).T
    best_policy = values_per_policy.argmax(axis=1)
    switch_idx = np.where(np.diff(best_policy) != 0)[0]
    return [np.array([w0, 1 - w0]) for w0 in w0_grid[switch_idx]]

# quick check: 2 policies (the segment endpoints) should give exactly 1 corner weight
demo_psi = [ground_truth_ccs[0], ground_truth_ccs[-1]]
print("Corner weight between the cheapest and priciest treasure:", corner_weights_2d(demo_psi))

Corner weight between the cheapest and priciest treasure: [array([0.462, 0.538])]


In [19]:
def estimate_improvement(w, psi_vecs, w_exp):
    """Delta(w) = v_bar*_w - v^SMP_w, via the LP in Algorithm 2."""
    if not w_exp:
        return np.inf  # nothing constrains the bound yet -> unbounded potential
    d = len(w)
    A_ub = np.array([w_prime for w_prime in w_exp])
    b_ub = np.array([v_smp(psi_vecs, w_prime) for w_prime in w_exp])
    res = linprog(c=-np.array(w), A_ub=A_ub, b_ub=b_ub, bounds=[(None, None)] * d, method="highs")
    v_bar = -res.fun if res.success else np.inf
    return v_bar - v_smp(psi_vecs, w)


# sanity check: with only the two segment endpoints solved, the improvement estimate
# at their exact corner weight should be strictly positive (there's a real gap to close)
w_corner = corner_weights_2d(demo_psi)[0]
print("Delta at the one corner weight:", estimate_improvement(w_corner, demo_psi, [np.array([1.0, 0.0]), np.array([0.0, 1.0])]))

Delta at the one corner weight: 8.808890436766786


In [20]:
def step_deterministic(s, a):
    ns = (s[0] + int(core.dir[a][0]), s[1] + int(core.dir[a][1]))
    if not (0 <= ns[0] <= 10 and 0 <= ns[1] <= 10) or core.sea_map[ns] == -10:
        ns = s
    treasure = core.sea_map[ns]
    terminal = treasure != 0 and treasure != -10
    phi = np.array([treasure if terminal else 0.0, -1.0])
    return ns, phi, terminal


nrows, ncols = core.sea_map.shape
states = [(r, c) for r in range(nrows) for c in range(ncols) if core.sea_map[r, c] != -10]
is_terminal = {s: core.sea_map[s] != 0 for s in states}
state_idx = {s: i for i, s in enumerate(states)}
n_states = len(states)
nonterminal = [s for s in states if not is_terminal[s]]


def policy_iteration(w, gamma=GAMMA):
    policy = {s: 0 for s in states if not is_terminal[s]}
    while True:
        A, b = np.eye(n_states), np.zeros(n_states)
        for s in states:
            if is_terminal[s]:
                continue
            i = state_idx[s]
            ns, phi, term = step_deterministic(s, policy[s])
            b[i] = phi @ w
            if not term:
                A[i, state_idx[ns]] -= gamma
        v = np.linalg.solve(A, b)
        V = {s: v[state_idx[s]] for s in states}

        stable = True
        for s in states:
            if is_terminal[s]:
                continue
            qs = []
            for a in range(N_ACTIONS):
                ns, phi, term = step_deterministic(s, a)
                qs.append(phi @ w + (0.0 if term else gamma * V[ns]))
            best_a = int(np.argmax(qs))
            stable &= best_a == policy[s]
            policy[s] = best_a
        if stable:
            return policy, V


def solve_task_exact(w, gamma=GAMMA):
    policy, _ = policy_iteration(w, gamma)
    A, b = np.eye(n_states), np.zeros((n_states, FEAT_DIM))
    for s in states:
        if is_terminal[s]:
            continue
        i = state_idx[s]
        ns, phi, term = step_deterministic(s, policy[s])
        b[i] = phi
        if not term:
            A[i, state_idx[ns]] -= gamma
    V = np.linalg.solve(A, b)
    V = {s: V[state_idx[s]] for s in states}

    psi = SFTable(N_ACTIONS, FEAT_DIM, np.zeros((N_ACTIONS, FEAT_DIM)))
    for s in states:
        if is_terminal[s]:
            continue
        table = np.zeros((N_ACTIONS, FEAT_DIM))
        for a in range(N_ACTIONS):
            ns, phi, term = step_deterministic(s, a)
            table[a] = phi if term else phi + gamma * V[ns]
        psi.table[s] = table
    return psi

In [21]:
def gpi_action(rng, state, w, sf_tables):
    """argmax_a max_i psi_i(state, a) . w, breaking ties uniformly at random."""
    values = np.stack([sft[state] @ w for sft in sf_tables], axis=0)  # (n_tables, A)
    action_values = values.max(axis=0)  # (A,) -- best over tables, per action
    max_val = action_values.max()
    candidates = np.flatnonzero(np.isclose(action_values, max_val))
    return int(rng.choice(candidates))


def solve_task(env, w, num_steps=100_000, alpha=0.1, gamma=GAMMA,
               eps_start=1.0, eps_end=0.05, optimistic_vec=(25.0, 0.0), seed=0):
    """Learn a tabular SF psi(s,a) for a single scalarized task r.w, via optimistic
    tabular TD-learning (Q-learning) -- the paper's other per-weight solver.
    optimistic_vec must upper-bound the true SF in every component so unvisited
    (state, action) pairs always look promising."""
    rng = random.Random(seed)
    default_vec = np.tile(np.array(optimistic_vec), (N_ACTIONS, 1))
    psi = SFTable(N_ACTIONS, FEAT_DIM, default_vec)

    obs, _ = env.reset(seed=seed)
    state = tuple(int(x) for x in obs)
    for t in range(num_steps):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * t / num_steps)
        if rng.random() < eps:
            action = rng.randrange(N_ACTIONS)
        else:
            action = gpi_action(rng, state, w, [psi])

        next_obs, phi, terminated, truncated, _ = env.step(action)
        next_state = tuple(int(x) for x in next_obs)
        if terminated:
            target = phi
        else:
            a_prime = gpi_action(rng, next_state, w, [psi])
            target = phi + gamma * psi[next_state][a_prime]
        psi[state][action] += alpha * (target - psi[state][action])

        if terminated or truncated:
            obs, _ = env.reset()
            state = tuple(int(x) for x in obs)
        else:
            state = next_state
    return psi


def solve_for_weight(w, solver="exact", num_steps=100_000, seed=0):
    """Dispatch a single per-weight solve to either the exact solver or Q-learning.
    Used everywhere a per-weight solve is needed: the standard baseline (Part 1) and
    the basis's own solves (Part 2 -- both the initial canonical corners and any
    Certify-or-Grow fallback). Q-learning here is genuinely noisy: even when a base
    policy it produces is imperfect, the Bellman certificate still checks against the
    exact known dynamics, so correctness never depends on solver quality -- a worse
    solve just costs more fallback solves, it can't silently certify a wrong answer."""
    if solver == "exact":
        return solve_task_exact(w)
    elif solver == "q_learning":
        return solve_task(env, w, num_steps=num_steps, seed=seed)
    raise ValueError(f"solver must be 'exact' or 'q_learning', got {solver!r}")

In [22]:
def own_greedy_actions(sf_table, w):
    return {s: int(np.argmax(sf_table[s] @ w)) for s in states if not is_terminal[s]}

In [23]:
def meta_policy_iteration_warm(w, Pi_base, W_base, gamma=GAMMA, gpi_init=True, max_sweeps=500):
    """max_sweeps is a safety cap, not a tuning knob: with EXACT base policies this
    loop always reaches a stable fixed point in a handful of sweeps (standard policy
    iteration theory -- finite deterministic policies, monotonic improvement). With
    Q-learning-based bases the own_actions/values feeding it are noisy, and near-ties
    between two base policies can make the greedy improvement step cycle between them
    forever instead of stabilizing. If the cap is hit, we return the current
    (possibly non-fixed-point) meta_policy/V as a best-effort result -- correctness
    downstream is preserved because sfols_certified's Bellman certificate checks
    whatever V comes back against the TRUE exact dynamics regardless of whether it's
    a genuine fixed point, so a capped-out V just means the certificate correctly
    reports a nonzero gap and falls back to a real solve, instead of an infinite loop."""
    own_actions = [own_greedy_actions(sft, w_j) for sft, w_j in zip(Pi_base, W_base)]
    if gpi_init:
        meta_policy = {}
        for s in nonterminal:
            vals = [Pi_base[j][s][own_actions[j][s]] @ w for j in range(len(Pi_base))]
            meta_policy[s] = int(np.argmax(vals))
    else:
        meta_policy = {s: 0 for s in nonterminal}

    n_sweeps = 0
    while True:
        n_sweeps += 1
        A, b = np.eye(n_states), np.zeros(n_states)
        for s in nonterminal:
            i = state_idx[s]
            ns, phi, term = step_deterministic(s, own_actions[meta_policy[s]][s])
            b[i] = phi @ w
            if not term:
                A[i, state_idx[ns]] -= gamma
        v = np.linalg.solve(A, b)
        V = {s: v[state_idx[s]] for s in states}
        stable = True
        for s in nonterminal:
            qs = []
            for j in range(len(Pi_base)):
                ns, phi, term = step_deterministic(s, own_actions[j][s])
                qs.append(phi @ w + (0.0 if term else gamma * V[ns]))
            best_j = int(np.argmax(qs))
            stable &= best_j == meta_policy[s]
            meta_policy[s] = best_j
        if stable or n_sweeps >= max_sweeps:
            return meta_policy, V, own_actions, n_sweeps


def meta_policy_sf(meta_policy, own_actions, gamma=GAMMA):
    A, b = np.eye(n_states), np.zeros((n_states, FEAT_DIM))
    for s in nonterminal:
        i = state_idx[s]
        ns, phi, term = step_deterministic(s, own_actions[meta_policy[s]][s])
        b[i] = phi
        if not term:
            A[i, state_idx[ns]] -= gamma
    Psi_arr = np.linalg.solve(A, b)
    return {s: Psi_arr[state_idx[s]] for s in states}

In [24]:
def bellman_gap(w, V_meta, gamma=GAMMA):
    gaps = {}
    for s in nonterminal:
        qs = []
        for a in range(N_ACTIONS):
            ns, phi, term = step_deterministic(s, a)
            qs.append(phi @ w + (0.0 if term else gamma * V_meta[ns]))
        gaps[s] = max(qs) - V_meta[s]
    return gaps


def sfols_certified(num_objectives=2, max_iterations=25, tol=1e-2, cert_tol=1e-6, verbose=True,
                     solver="exact", num_steps=100_000, seed=0):
    Pi_base, W_base = [], []
    for i in range(num_objectives):
        w = np.zeros(num_objectives)
        w[i] = 1.0
        Pi_base.append(solve_for_weight(w, solver=solver, num_steps=num_steps, seed=seed + i))
        W_base.append(w)

    Psi, W_exp = [], []
    counter = itertools.count()
    Q = []
    for i in range(num_objectives):
        w = np.zeros(num_objectives)
        w[i] = 1.0
        heapq.heappush(Q, (-np.inf, next(counter), tuple(w)))

    history = []
    n_composed, n_fallback = 0, 0
    it = 0
    while Q and it < max_iterations:
        neg_priority, _, w_tuple = heapq.heappop(Q)
        priority = -neg_priority
        if priority < tol and Psi:
            break
        w = np.array(w_tuple)
        it += 1

        meta_policy, V_meta, own_actions, _ = meta_policy_iteration_warm(w, Pi_base, W_base)
        worst_gap = max(bellman_gap(w, V_meta).values())

        if worst_gap > cert_tol:
            n_fallback += 1
            psi_table = solve_for_weight(w, solver=solver, num_steps=num_steps, seed=seed + it)
            Pi_base.append(psi_table)
            W_base.append(w)
            own_w = own_greedy_actions(psi_table, w)
            psi_vec = psi_table[START_STATE][own_w[START_STATE]]
            status = f"FALLBACK (gap={worst_gap:.4f}) -- basis grew to {len(Pi_base)}"
        else:
            n_composed += 1
            psi_vec = meta_policy_sf(meta_policy, own_actions)[START_STATE]
            status = "composed"

        W_exp.append(w)
        is_new = not any(np.allclose(psi_vec, v, atol=1e-1) for v in Psi)

        if verbose:
            print(f"iter {it:2d}  w={w.round(3)}  priority={priority:8.4f}  psi={psi_vec.round(3)}  new={is_new}  [{status}]")

        if is_new:
            kept = []
            for (p, c, wt) in Q:
                if p == -np.inf or not (psi_vec @ np.array(wt) > v_smp(Psi, np.array(wt)) + 1e-9):
                    kept.append((p, c, wt))
            Q = kept
            heapq.heapify(Q)

            corners = corner_weights_2d(Psi + [psi_vec]) if Psi else []
            Psi.append(psi_vec)

            for wc in corners:
                delta = estimate_improvement(wc, Psi, W_exp)
                heapq.heappush(Q, (-delta, next(counter), tuple(wc)))

        history.append((it, len(Psi), priority))

    return Pi_base, W_base, Psi, W_exp, history, n_composed, n_fallback


Pi_base_c, W_base_c, Psi_c, W_exp_c, history_c, n_composed, n_fallback = sfols_certified(
    max_iterations=25, tol=1e-2, verbose=True)

print(f"\ndiscovered {len(Psi_c)} points (ground-truth CCS has {len(ground_truth_ccs)} points)")
print(f"final basis size: {len(Pi_base_c)}  (composed: {n_composed}, fallback exact solves: {n_fallback})")

Psi_c_arr = np.array(Psi_c)
all_matched = True
for gt in ground_truth_ccs:
    dists = np.linalg.norm(Psi_c_arr - gt, axis=1)
    best = dists.argmin()
    matched = dists[best] < 1e-6
    all_matched &= matched
print(f"all {len(ground_truth_ccs)} ground-truth CCS points recovered exactly: {all_matched}")

iter  1  w=[1. 0.]  priority=     inf  psi=[ 19.778 -17.383]  new=True  [composed]
iter  2  w=[0. 1.]  priority=     inf  psi=[ 0.7 -1. ]  new=True  [composed]
iter  3  w=[0.462 0.538]  priority=  8.8089  psi=[11.047 -4.901]  new=True  [composed]
iter  4  w=[0.588 0.412]  priority=  2.0529  psi=[14.856 -8.648]  new=True  [composed]
iter  5  w=[0.274 0.726]  priority=  1.5869  psi=[ 8.037 -2.97 ]  new=True  [composed]
iter  6  w=[0.639 0.361]  priority=  0.6100  psi=[ 17.373 -12.248]  new=True  [composed]
iter  7  w=[0.212 0.788]  priority=  0.4459  psi=[ 0.7 -1. ]  new=False  [composed]
iter  8  w=[0.212 0.788]  priority=  0.4459  psi=[ 0.7 -1. ]  new=False  [composed]
iter  9  w=[0.681 0.319]  priority=  0.2768  psi=[ 19.073 -15.706]  new=True  [composed]
iter 10  w=[0.391 0.609]  priority=  0.2181  psi=[ 8.037 -2.97 ]  new=False  [composed]
iter 11  w=[0.391 0.609]  priority=  0.2181  psi=[ 8.037 -2.97 ]  new=False  [composed]
iter 12  w=[0.391 0.609]  priority=  0.2181  psi=[ 8.037 

## Part 3 (new): Envelope GPI

Construction from `notes/adaptive_basis.tex` (Definition `def:envelope`, Theorem
`thm:envelope`): each base policy's own scalar value `V^{pi_j}(s) = Psi_j(s,pi_j(s)).w`,
`U_B(s) = max(V_meta(s), max_j V^{pi_j}(s))`, `pi_star` = one full-action Bellman
backup off `U_B`. Reuses `meta_policy_iteration_warm` (already computes exactly the
own-actions and own-values Certify-or-Grow needs for its certificate) -- envelope-GPI's
only new step is the `max` with `V_meta` and one extra backup, the same shape as
`pi^hyb` (Proposition `prop:hyb`).

In [25]:
## Envelope GPI on the REAL, certified DST basis (Pi_base_c, W_base_c from Part 2 above)
##
## Reuses exactly the machinery already built for Certify-or-Grow: own_greedy_actions,
## meta_policy_iteration_warm (which already computes each base policy's own action AND
## its own scalar value at the query weight internally). The only NEW piece is combining
## V_meta with max_j V_own_j into U_B, then one extra Bellman backup -- the same shape as
## pi^hyb (Proposition prop:hyb), just off U_B instead of off V_meta alone.

def own_values_at_w(w, Pi_base, W_base):
    """V^{pi_j}(s) := Psi_j(s, pi_j(s)) . w for every base policy j, every state.
    pi_j's OWN action only depends on W_base[j] (fixed once solved), never on the query w."""
    own_actions = [own_greedy_actions(sft, w_j) for sft, w_j in zip(Pi_base, W_base)]
    V_own = np.zeros((len(Pi_base), n_states))
    for j, (sft, oa) in enumerate(zip(Pi_base, own_actions)):
        for s in nonterminal:
            V_own[j, state_idx[s]] = sft[s][oa[s]] @ w
    return own_actions, V_own


def envelope_pi_star(w, Pi_base, W_base, gamma=GAMMA):
    meta_policy, V_meta, own_actions, _ = meta_policy_iteration_warm(w, Pi_base, W_base, gamma=gamma)
    _, V_own = own_values_at_w(w, Pi_base, W_base)
    V_meta_arr = np.array([V_meta[s] for s in states])
    U = np.maximum(V_meta_arr, V_own.max(axis=0))
    U_dict = {s: U[state_idx[s]] for s in states}

    pi_star = {}
    Q_U = {}
    for s in nonterminal:
        qs = []
        for a in range(N_ACTIONS):
            ns, phi, term = step_deterministic(s, a)
            qs.append(phi @ w + (0.0 if term else gamma * U_dict[ns]))
        Q_U[s] = np.array(qs)
        pi_star[s] = int(np.argmax(qs))
    return pi_star, Q_U, U_dict, meta_policy, V_meta


def evaluate_scalar_policy(policy, w, gamma=GAMMA):
    """Real policy evaluation on the TRUE exact DST dynamics -- the honest, realized value."""
    A, b = np.eye(n_states), np.zeros(n_states)
    for s in nonterminal:
        i = state_idx[s]
        ns, phi, term = step_deterministic(s, policy[s])
        b[i] = phi @ w
        if not term:
            A[i, state_idx[ns]] -= gamma
    v = np.linalg.solve(A, b)
    return {s: v[state_idx[s]] for s in states}


def qhat_B_full(w, Pi_base):
    """Standard GPI's own pool estimate: max_j Psi_j(s,a).w, for EVERY (s,a) -- the
    expensive O(k|A|d) full-table construction envelope-GPI is meant to avoid."""
    Qhat = {}
    pi_gpi = {}
    for s in nonterminal:
        vals = np.stack([sft[s] @ w for sft in Pi_base], axis=0)  # (k, A)
        best_over_pool = vals.max(axis=0)  # (A,)
        Qhat[s] = best_over_pool
        pi_gpi[s] = int(np.argmax(best_over_pool))
    return Qhat, pi_gpi


print("Envelope-GPI helpers defined; using the REAL certified DST basis "
      f"(k={len(Pi_base_c)} base policies) from Part 2/3 above.")


Envelope-GPI helpers defined; using the REAL certified DST basis (k=2 base policies) from Part 2/3 above.


## Part 4: Theorem check on the real domain (`Q^{pi_star} >= Qhat_B`, every `(s,a)`)

In [26]:
## Theorem check on the REAL domain: Q^{pi_star}(s,a) >= Qhat_B(s,a) for every (s,a),
## at a grid of NEW query weights (never used to build the basis).

rng = np.random.default_rng(0)
test_weights = np.stack([rng.dirichlet(np.ones(FEAT_DIM)) for _ in range(200)], axis=0)

worst_violation = -np.inf
worst_margin = -np.inf
n_violations = 0

for w in test_weights:
    pi_star, Q_U, U_dict, meta_policy, V_meta = envelope_pi_star(w, Pi_base_c, W_base_c)
    Qhat, pi_gpi = qhat_B_full(w, Pi_base_c)
    for s in nonterminal:
        gap = Q_U[s] - Qhat[s]  # >= 0 everywhere is the claim
        worst_violation = max(worst_violation, np.max(-gap))
        worst_margin = max(worst_margin, np.max(gap))
        if np.max(-gap) > 1e-6:
            n_violations += 1

print(f"Theorem check (Q_pi_star >= Qhat_B) on REAL DST, {len(test_weights)} random test weights, "
      f"{len(nonterminal)} states each:")
print(f"  violations: {n_violations}  |  worst violation: {worst_violation:.8f}  |  "
      f"largest positive margin found: {worst_margin:.6f}")


Theorem check (Q_pi_star >= Qhat_B) on REAL DST, 200 random test weights, 62 states each:
  violations: 0  |  worst violation: 0.00000000  |  largest positive margin found: 2.623464


## Part 5: Honest realized-value comparison

The theorem only proves `pi_star` matches GPI's *estimate*. This checks, on the real
domain, whether GPI's own *realized* value ever beats `pi_star`'s -- exactly the
open question from `SFOLS/11`'s random-MDP experiment (there: 16/500 and 15/500).

In [12]:
## Honest realized-value comparison on REAL DST, at START_STATE, across many test weights:
## pi_star (envelope) vs pi^GPI_B (realized) vs pi_meta (realized) vs true V*_w (exact optimum).

rows = []
for w in test_weights:
    pi_star, Q_U, U_dict, meta_policy, V_meta = envelope_pi_star(w, Pi_base_c, W_base_c)
    Qhat, pi_gpi = qhat_B_full(w, Pi_base_c)

    V_star_realized = evaluate_scalar_policy(pi_star, w)
    V_gpi_realized = evaluate_scalar_policy(pi_gpi, w)
    _, V_true = policy_iteration(w)  # true exact optimum for this weight

    rows.append((
        w.copy(),
        V_star_realized[START_STATE],
        V_gpi_realized[START_STATE],
        V_meta[START_STATE],
        V_true[START_STATE],
    ))

W_arr = np.stack([r[0] for r in rows])
V_star_all = np.array([r[1] for r in rows])
V_gpi_all = np.array([r[2] for r in rows])
V_meta_all = np.array([r[3] for r in rows])
V_true_all = np.array([r[4] for r in rows])

gpi_minus_star = V_gpi_all - V_star_all  # >0 means GPI's REALIZED value beats pi_star's
n_gpi_wins = int(np.sum(gpi_minus_star > 1e-6))
n_star_wins = int(np.sum(gpi_minus_star < -1e-6))
n_ties = len(rows) - n_gpi_wins - n_star_wins

print(f"Realized value at START_STATE, {len(rows)} test weights on REAL DST "
      f"(k={len(Pi_base_c)} certified base policies):")
print(f"  pi_star  regret vs true optimum: mean={np.mean(V_true_all - V_star_all):.5f}  "
      f"max={np.max(V_true_all - V_star_all):.5f}")
print(f"  pi^GPI_B regret vs true optimum: mean={np.mean(V_true_all - V_gpi_all):.5f}  "
      f"max={np.max(V_true_all - V_gpi_all):.5f}")
print(f"  pi_meta  regret vs true optimum: mean={np.mean(V_true_all - V_meta_all):.5f}  "
      f"max={np.max(V_true_all - V_meta_all):.5f}")
print()
print(f"  GPI's realized value beats pi_star's realized value: {n_gpi_wins}/{len(rows)} weights "
      f"(worst gap {np.max(gpi_minus_star):.5f})")
print(f"  pi_star's realized value beats GPI's realized value: {n_star_wins}/{len(rows)} weights "
      f"(worst gap {np.max(-gpi_minus_star):.5f})")
print(f"  exact ties: {n_ties}/{len(rows)}")


Realized value at START_STATE, 200 test weights on REAL DST (k=2 certified base policies):
  pi_star  regret vs true optimum: mean=0.00000  max=0.00000
  pi^GPI_B regret vs true optimum: mean=0.00767  max=0.29142
  pi_meta  regret vs true optimum: mean=0.00000  max=0.00000

  GPI's realized value beats pi_star's realized value: 0/200 weights (worst gap 0.00000)
  pi_star's realized value beats GPI's realized value: 11/200 weights (worst gap 0.29142)
  exact ties: 189/200


## Part 6: Real wall-clock cost per new query weight

In [14]:
## Real wall-clock cost, per NEW query weight, on the REAL certified DST basis.
## "Standard GPI cost" = building the full pool estimate Qhat_B(s,a) for every (s,a)
## (the O(k|A|d) construction). "Envelope cost" = own-value scalars (O(kd) total, no
## per-action loop) + the metapolicy's restricted PI (already paid by Certify-or-Grow
## regardless) + one full-action Bellman backup off U_B (O(|A|d), shared across all k).

N_TIMING_WEIGHTS = 300
timing_weights = [rng.dirichlet(np.ones(FEAT_DIM)) for _ in range(N_TIMING_WEIGHTS)]

t0 = time.perf_counter()
for w in timing_weights:
    qhat_B_full(w, Pi_base_c)
t_gpi = time.perf_counter() - t0

t0 = time.perf_counter()
for w in timing_weights:
    envelope_pi_star(w, Pi_base_c, W_base_c)
t_envelope = time.perf_counter() - t0

# isolate the part of envelope-GPI that's genuinely NEW (i.e. excluding the restricted-PI
# solve that Certify-or-Grow already has to run anyway to get V_meta / the certificate)
t0 = time.perf_counter()
for w in timing_weights:
    meta_policy_iteration_warm(w, Pi_base_c, W_base_c)
t_meta_only = time.perf_counter() - t0
t_envelope_incremental = t_envelope - t_meta_only

print(f"k={len(Pi_base_c)} base policies, |A|={N_ACTIONS}, d={FEAT_DIM}, "
      f"|S|={n_states} states, {N_TIMING_WEIGHTS} new query weights.\n")
print(f"standard GPI full-pool estimate (Qhat_B, all s,a):  {t_gpi*1000:.1f} ms total, "
      f"{t_gpi/N_TIMING_WEIGHTS*1000:.3f} ms/query")
print(f"envelope GPI (pi_star), end-to-end incl. meta-PI:   {t_envelope*1000:.1f} ms total, "
      f"{t_envelope/N_TIMING_WEIGHTS*1000:.3f} ms/query")
print(f"  of which restricted meta-PI alone (already paid \n"
      f"  by Certify-or-Grow regardless of envelope-GPI):   {t_meta_only*1000:.1f} ms total")
print(f"  envelope-GPI's OWN incremental cost on top of\n"
      f"  Certify-or-Grow's existing metapolicy machinery:  {t_envelope_incremental*1000:.1f} ms total, "
      f"{t_envelope_incremental/N_TIMING_WEIGHTS*1000:.3f} ms/query")
print()
print(f"speedup, full construction:            {t_gpi/t_envelope:.2f}x")
print(f"speedup, incremental cost only:        {t_gpi/max(t_envelope_incremental, 1e-9):.2f}x")


k=2 base policies, |A|=4, d=2, |S|=72 states, 300 new query weights.

standard GPI full-pool estimate (Qhat_B, all s,a):  309.1 ms total, 1.030 ms/query
envelope GPI (pi_star), end-to-end incl. meta-PI:   1424.0 ms total, 4.747 ms/query
  of which restricted meta-PI alone (already paid 
  by Certify-or-Grow regardless of envelope-GPI):   872.8 ms total
  envelope-GPI's OWN incremental cost on top of
  Certify-or-Grow's existing metapolicy machinery:  551.2 ms total, 1.837 ms/query

speedup, full construction:            0.22x
speedup, incremental cost only:        0.56x
